# 02 · Candidate generation (blocking)

For every S1 we retrieve S2/S3 records **of the same country** through independent sparse TF-IDF channels, each giving its top-K by cosine (`sparse_dot_topn`, chunked — never an all-pairs matrix):

| channel | representation | catches |
|---|---|---|
| `addr` | address word tokens | renamed / DBA / truncated names at the same address |
| `both` | name words ⊕ address words | common names disambiguated by place |
| `cross` | name-token × address-token conjunctions | one business inside a building shared by many |
| `name` | name word tokens | same business, reformatted or different address |
| `noaddr` | name word tokens, searched only among address-less targets | the 3–4 % of records without an address |
| `translit` | phonetic name × address conjunctions, searched only among non-Latin-script targets | Devanagari names romanized without vowels (`laiph phainens` = life finance) |

Every union pair gets the exact cosine of **every** channel plus its rank in each channel (−1 = not proposed); these become features. A stage-0 re-ranker then trims the union to the best N per S1.

Why several channels (EXP-002…005 in `docs/EXPERIMENTS.md`): on India, name-only retrieval reaches ~0.50 recall@40 (generic, duplicated names) and address-only ~0.85@40 (shared buildings); they fail on different pairs, and address ∪ both ∪ cross reaches 0.964@40; the `noaddr` and `translit` channels lift India to 0.985 (US: 0.991). Character n-gram and phonetic channels were tested and dropped for retrieval (lower recall, slower); the phonetic key stays as a feature.

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

In [ ]:
for ch in stages.make_channels(S):
    print(f"{ch.name:6s} top_k={ch.top_k:3d} max_df_frac={ch.max_df_frac} fields={[(f.column, f.kind, f.weight) for f in ch.fields]}")

## Train + test candidates (raw union)

In [ ]:
raw_stats = pl.concat([stages.run_candidates(S, "train"), stages.run_candidates(S, "test")])
raw_stats

In [ ]:
raw_report = stages.candidate_report(S, "train", which="raw")
raw_report.select("country", "channel", "pair_recall", "avg_per_s1", "p95_per_s1")

## Stage-0 pruning

A small LightGBM on the retrieval-only features (per-channel cosine/rank, query and target context) re-ranks the union; the best `max_candidates` per S1 are kept. This is the candidate set the matching model scores and what goes into `candidate_pairs.tsv`.

Pick `max_candidates` from the `topN` rows below: the smallest N whose recall is within ~0.1 pt of the raw union.

In [ ]:
prune_stats = stages.run_prune(S)
prune_stats

In [ ]:
report = stages.candidate_report(S, "train", which="pruned", cutoffs=(10, 20, 30, 40, 50))
report.select("country", "channel", "pair_recall", "avg_per_s1", "p95_per_s1", "p99_per_s1")

`pair_recall` of the pruned `union` is the **recall ceiling**: a pair missing here can never be predicted. Record the numbers in `docs/EXPERIMENTS.md`.

## Optional: recall-vs-K sweep for one channel on a sample

Use this to justify `top_k_*` settings. It re-fits one channel on a sample of S1 against the full target pool of a country.

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    from entity_forge.blocking import build_index, topk_search
    from entity_forge.io import read_ground_truth_pairs
    ckey, n_queries, channel = "india", 20_000, stages.make_channels(S)[3]
    q, t = stages.load_records(S, "train", ckey)
    q = q.sample(min(n_queries, q.height), seed=0)
    truth = read_ground_truth_pairs(S.ground_truth).join(q.select(pl.col("entity_id").alias("s1_id")), on="s1_id", how="semi")
    hits = topk_search(build_index(channel, q, t), top_k=100)
    hits = hits.with_columns(q["entity_id"].gather(hits["q"]).alias("s1_id"), t["entity_id"].gather(hits["t"]).alias("t_id"))
    display(pl.DataFrame([{"K": k, "recall": truth.join(hits.filter(pl.col("rank") < k), on=["s1_id", "t_id"], how="semi").height / truth.height}
                          for k in (5, 10, 20, 40, 60, 100)]))